# Neuro101DD — Data Analysis Tutorial

## Setup

Run the first two cells every time you open this notebook or the session times out. 
The third cell is a one-off check that everything works — please run it early in the week.


In [ ]:
#RUN THIS FIRST -- takes about a minute, and only needs doing once per session.


!pip -q install "spikeinterface==0.104.9" "probeinterface==0.4.0" "neo==0.14.5"
!pip -q install "tridesclous==1.6.9"
print("install finished")


In [ ]:
#RUN THIS SECOND, every time you open or reset the session.

import os, sys, pathlib, subprocess
import numpy as np

for _old, _new in [("in1d", "isin"), ("alltrue", "all"), ("sometrue", "any"),
                   ("row_stack", "vstack"), ("trapz", "trapezoid")]:
    if not hasattr(np, _old) and hasattr(np, _new):
        setattr(np, _old, getattr(np, _new))
for _old, _new in [("float_", "float64"), ("int_", "int64"), ("bool8", "bool_"),
                   ("unicode_", "str_"), ("NaN", "nan"), ("Inf", "inf"),
                   ("infty", "inf"), ("NINF", "nan")]:
    if not hasattr(np, _old) and hasattr(np, _new):
        try:
            setattr(np, _old, getattr(np, _new))
        except Exception:
            pass

UTILS_URL = ("https://raw.githubusercontent.com/mhburrell/Neuro101DD_Tutorial/"
             "refs/heads/main/n101dd_functions.py")
import hashlib
subprocess.check_call([
    "curl", "-sSL", "--fail",
    "-H", "Cache-Control: no-cache", "-H", "Pragma: no-cache",
    UTILS_URL, "-o", "workshop_utils.py",
])
_b = pathlib.Path("workshop_utils.py").read_bytes()
print(f"  workshop_utils.py: {len(_b):,} bytes  sha {hashlib.sha256(_b).hexdigest()[:8]}")
sys.path.insert(0, str(pathlib.Path(".").resolve()))
sys.modules.pop("workshop_utils", None)

from workshop_utils import (plot_trial, plot_all_trials, listen_recording,
                            filter_recording, run_sorting, load_recording)

import spikeinterface, tridesclous
print(f"python        {sys.version.split()[0]}")
print(f"numpy         {np.__version__}")
print(f"spikeinterface{spikeinterface.__version__:>10}")
print(f"tridesclous   {tridesclous.__version__:>10}")
print(f"np.in1d available: {hasattr(np, 'in1d')}")
print("\nsetup ok")


In [ ]:
#OPTIONAL BUT RECOMMENDED -- run this once, today, not the night before the due date.
#
# This does a complete miniature version of everything the assignment asks for,
# on made-up data, in about ten seconds. If it prints "SELF-TEST PASSED" your
# environment is fine. If it does not, email now rather than later.

import numpy as np, spikeinterface as si
import spikeinterface.sorters as ss, spikeinterface.preprocessing as sp

rng = np.random.default_rng(0)
_fs = 22000.0
_t = np.linspace(0, 1, 57)
_wave = (0.30*np.exp(-((_t-0.12)**2)/(2*0.06**2)) - np.exp(-((_t-0.28)**2)/(2*0.07**2))
         + 0.55*np.exp(-((_t-0.62)**2)/(2*0.20**2)))
_wave = _wave / np.abs(_wave).max()
_segs = []
for _ in range(10):
    x = rng.normal(0, 0.09, int(2*_fs))
    for ts in np.sort(rng.uniform(0.02, 1.9, 12)):
        i0 = int(ts*_fs)
        x[i0:i0+len(_wave)] += 0.6*_wave
    _segs.append(x.astype("float32")[:, None])

_rec = si.NumpyRecording(_segs, sampling_frequency=_fs)
_rec.set_dummy_probe_from_locations(np.zeros((1, 2)))
_rec = _rec.save(folder="selftest_data", overwrite=True, progress_bar=False)
_filt = sp.bandpass_filter(_rec, freq_min=300, freq_max=6000)
_sorting = ss.run_sorter("tridesclous", _filt, folder="selftest_out",
                         remove_existing_folder=True, detect_threshold=5, verbose=False)
_n = sum(len(_sorting.get_unit_spike_train(unit_id=u, segment_index=k))
         for u in _sorting.get_unit_ids() for k in range(_rec.get_num_segments()))
print(f"units found: {len(_sorting.get_unit_ids())}   spikes found: {_n} (about 120 expected)")
assert len(_sorting.get_unit_ids()) >= 1 and _n > 60, "sorting did not work -- email Mark"
print("\nSELF-TEST PASSED -- filtering and spike sorting both work in this session.")


In [ ]:
#RUN THIS THIRD -- downloads YOUR ten recordings (about 160 MB, a minute or so).
# Your code is already filled in below. It is not the same as anyone else's, and
# neither are your recordings, so do not copy this line from another notebook.

STUDENT_CODE = ""          # <-- filled in for you

import pathlib, subprocess, zipfile

def _in_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True

if not _in_colab():
    raise RuntimeError(
        "This notebook fetches your data only when it is running in Google Colab, and "
        "this is not Colab.\n\n"
        "Open the notebook at https://colab.research.google.com and run it there.\n\n"
        "If you ARE in Colab and seeing this message, something has changed at Google's "
        "end -- email me and I will fix it. Please do not edit this cell to get around it, "
        "because the data you would be working on then is not yours."
    )

assert STUDENT_CODE, (
    "STUDENT_CODE is empty. This is the master copy of the notebook, not yours.\n"
    "Open the notebook that was shared with you -- your code is already in it."
)
URL = f"https://github.com/mhburrell/Neuro101DD_Tutorial/releases/download/Data2026/{STUDENT_CODE}.zip"
dest = pathlib.Path("/content/data")
if not any(dest.glob("recording*_events.json")):
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["curl", "-sSL", "--fail", URL, "-o", "/content/mydata.zip"])
    with zipfile.ZipFile("/content/mydata.zip") as z:
        z.extractall(dest)
    inner = [p for p in dest.iterdir() if p.is_dir() and (p / "dataset_id.txt").exists()]
    if inner:                                  # flatten a single wrapping folder
        for item in inner[0].iterdir():
            item.rename(dest / item.name)

print("recordings available:",
      sorted(int(p.name.replace("recording", "").replace("_events.json", ""))
             for p in dest.glob("recording*_events.json")))
stamp = dest / "dataset_id.txt"
if stamp.exists():
    print("\ndata set:", stamp.read_text().strip())
    print("Paste that line at the top of your answer to Q2.1.")


# Data Inspection

You have ten recordings. Three functions let you look at the raw data before you do anything to it.

Your ten recordings are not the same as anyone else's, so the numbers you report should be your own.


## plot_all_trials

Plots every trial in one recording on the same axes, each trial a different color. 
Useful for spotting anything that changes partway through the recording.


In [ ]:
plot_all_trials(recording_number=1)
#Change the recording number (1 to 10) and re-run to compare recordings.


## plot_trial

Plots a single trial, so you can see individual spikes. Comparing an early trial 
(1, 2, 3) with a late one (100, 120) is often informative.


In [ ]:
plot_trial(recording_number=1, trial_num=1)


## listen_recording

Plays a few trials as audio, with a one-second gap between them. Spikes sound like clicks. 
Have a listen to each recording — some problems are much more obvious by ear than by eye.


In [ ]:
listen_recording(recording_number=1, start_trial=1)
#This plays five trials starting from start_trial.


# Filtering

A raw recording contains spikes and also a lot of things that are not spikes: slow drift, 
movement, and electrical interference from the mains. Filtering keeps the frequency band 
where spikes live and discards the rest.

This function plots part of the trace and its power spectrum, before and after filtering. 
Look at the power spectrum of the **raw** trace first: the recording was already filtered 
once, when it was made, and you can read the band it was filtered to off the plot. 
Look for where the power drops away sharply at each end.


In [ ]:
filter_recording(recording_number=1, freq_min=300, freq_max=6000)
#Values worth trying -- freq_min: 100, 250, 500, 1000   freq_max: 2000, 5000, 8000


# Spike Sorting

`run_sorting` does two things. First it detects candidate spikes, wherever the trace crosses 
a threshold. The threshold is set in units of 
[median absolute deviation](https://en.wikipedia.org/wiki/Median_absolute_deviation) of the 
trace, so `detect_threshold=5` means five MADs from the median. Then it sorts those detected 
spikes into units by the shape of the waveform, using principal component analysis.

It prints the average firing rate of each unit, and plots the template waveform, a stretch of 
trace with the detected spikes marked, and a raster and PSTH for each unit.

Two things to be careful about. A threshold set too low will call noise a spike; set too high, 
it will miss real spikes. And the choice is not only about how many spikes you get — it can 
change what you conclude at the end of this assignment, so keep a note of the value you used 
and why.


In [ ]:
run_sorting(recording_number=1, detect_threshold=5, filter=True, freq_min=300, freq_max=6000)
#Values worth trying for detect_threshold: 3, 4, 5, 6, 7, 9
#Some recordings are noisy enough that you will need filter=True.
